In [1]:
import pandas as pd
import numpy as np
import string
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Abusive-Text-dravidianLangtech/train2.csv')

In [4]:
def preprocess_tamil(text):
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text
df['cleaned_text'] = df['Text'].apply(preprocess_tamil)

In [5]:
X = df['cleaned_text']
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    max_features=15000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [7]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [ ]:
svm_model = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')
svm_model.fit(X_train_tfidf, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 1. Predict on the validation set
y_val_pred = svm_model.predict(X_test_tfidf)

# 2. Print the report
print("### Performance Metrics ###")
print(classification_report(y_test, y_val_pred))

# 3. Specifically check Macro F1 (This is your main competition metric)
macro_f1 = f1_score(y_test, y_val_pred, average='macro')
print(f"Your Current Macro F1 Score: {macro_f1:.4f}")

### Performance Metrics ###
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      2673
           1       0.87      0.85      0.86      2517

    accuracy                           0.87      5190
   macro avg       0.87      0.86      0.87      5190
weighted avg       0.87      0.87      0.87      5190

Your Current Macro F1 Score: 0.8651


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_val_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Abusive', 'Abusive'], yticklabels=['Non-Abusive', 'Abusive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: KEC_Inference Model')
plt.show()

NameError: name 'y_test' is not defined

In [ ]:
import zipfile
from google.colab import files

# 1. Load the official hidden test set (ensure the path is correct)
test_df = pd.read_csv('/content/drive/MyDrive/Abusive-Text-dravidianLangtech/TestV2 - testV2.csv')

# 2. Preprocess and Predict (Labels will be 0 and 1)
X_official_test = tfidf.transform(test_df['Text'].apply(preprocess_tamil))
final_preds = svm_model.predict(X_official_test)

# 3. Create the CSV (Naming: KEC_LangTech.csv)
submission_df = pd.DataFrame({
    'Text': test_df['Text'],
    'Class': final_preds
})
csv_name = "KEC_LangTech_Run1.csv"
submission_df.to_csv(csv_name, index=False)

In [ ]:
y_pred = svm_model.predict(X_test_tfidf)

print("### Evaluation Results for Paper ###")
print(classification_report(y_test, y_pred))

### Evaluation Results for Paper ###
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      2673
           1       0.87      0.85      0.86      2517

    accuracy                           0.87      5190
   macro avg       0.87      0.86      0.87      5190
weighted avg       0.87      0.87      0.87      5190



In [ ]:
test_sentences = [
    "இன்று நல்ல நாள்",
    "நீ ஒரு முட்டாள்"
]

In [ ]:
cleaned_test_sentences = [preprocess_tamil(sentence) for sentence in test_sentences]

In [ ]:
X_custom_tfidf = tfidf.transform(cleaned_test_sentences)

In [ ]:
custom_predictions = svm_model.predict(X_custom_tfidf)

In [ ]:
for sentence, label in zip(test_sentences, custom_predictions):
    sentiment = "Abusive (1)" if label == 1 else "Non-Abusive (0)"
    print(f"Text: {sentence}")
    print(f"Prediction: {sentiment}\n")

Text: இன்று நல்ல நாள்
Prediction: Non-Abusive (0)

Text: நீ ஒரு முட்டாள்
Prediction: Abusive (1)



In [ ]:
import pandas as pd
import zipfile
import os

In [ ]:
test_df = pd.read_csv('/content/drive/MyDrive/Abusive-Text-dravidianLangtech/TestV2 - testV2.csv')

In [ ]:
X_test_final = tfidf.transform(test_df['Text'].apply(preprocess_tamil))
final_predictions = svm_model.predict(X_test_final)

In [ ]:
team_name = "KECInference_Run2"
csv_filename = f"{team_name}.csv"

# Ensure the CSV has the columns expected by the organizers (usually 'Text' and 'Class')
submission_df = pd.DataFrame({
    'Text': test_df['Text'],
    'Class': final_predictions
})

submission_df.to_csv(csv_filename, index=False)
print(f"✅ Created CSV: {csv_filename}")

✅ Created CSV: KECInference_Run2.csv
